# Comparación de algoritmos de aprendizaje supervisado

- **Materia:** Fundamentos de Inteligencia Artificial
- **Docente:** Ing. Carlos André Castro Rodríguez
- **Universidad:** Universidad Tecnológica de Cancún · LITIID71
- **Equipo:**
  - Navarrete Tzuc Jorge
  - Ríos Astudillo Daniel Eduardo
  - Itza Santizo Jarett Osiel
  - Salinas Beatriz Erick

**Objetivo.** Entrenar KNN, SVM y una red neuronal (MLP) con el mismo conjunto de datos y la misma división, y decidir cuál conviene para clasificar tumores como benignos o malignos.

**Decisión que guía todo el análisis.** La clase positiva es **maligno**. Precisión, exhaustividad y F1 se calculan sobre esa clase, porque el error más grave aquí es decirle "benigno" a un tumor maligno.

## 1. Librerías y configuración

In [1]:
import sys
import time
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from matplotlib.colors import LinearSegmentedColormap
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEMILLA = 42
MODELOS = ["KNN", "SVM", "MLP"]
METRICAS = ["Accuracy", "Precision", "Recall", "F1-score"]

# Cada modelo conserva su color y su marcador en todas las gráficas.
COLOR = {"KNN": "#2a78d6", "SVM": "#eb6834", "MLP": "#1baf7a"}
MARCADOR = {"KNN": "o", "SVM": "s", "MLP": "D"}
FONDO, TINTA, TINTA_SUAVE, REJILLA = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "figure.facecolor": FONDO,
        "axes.facecolor": FONDO,
        "axes.edgecolor": REJILLA,
        "axes.labelcolor": TINTA_SUAVE,
        "axes.titlecolor": TINTA,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "grid.color": REJILLA,
        "xtick.color": TINTA_SUAVE,
        "ytick.color": TINTA_SUAVE,
        "legend.frameon": False,
    }
)

print("Python      ", sys.version.split()[0])
print("scikit-learn", sklearn.__version__)
print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("matplotlib  ", matplotlib.__version__)

Python       3.13.15
scikit-learn 1.9.1
pandas       3.1.1
numpy        2.5.4
matplotlib   3.11.2


## 2. Carga del dataset

Usamos *Breast Cancer Wisconsin (Diagnostic)*, el archivo adjunto de la actividad. `wdbc.data` no trae encabezados, así que los nombres salen de la documentación del dataset (`wdbc.names`).

El notebook busca el archivo en `data/`. Si no lo encuentra, como pasa al abrirlo en Colab, lo descarga del repositorio del equipo.

In [2]:
RUTA_LOCAL = Path("data/wdbc.data")
URL_REPO = (
    "https://raw.githubusercontent.com/jorgemx546/"
    "comparacion-aprendizaje-supervisado/main/data/wdbc.data"
)

MEDIDAS = [
    "radius", "texture", "perimeter", "area", "smoothness",
    "compactness", "concavity", "concave_points", "symmetry", "fractal_dimension",
]
RESUMENES = ["mean", "se", "worst"]
CARACTERISTICAS = [f"{m}_{r}" for r in RESUMENES for m in MEDIDAS]

origen = RUTA_LOCAL if RUTA_LOCAL.exists() else URL_REPO
datos = pd.read_csv(origen, header=None, names=["id", "diagnosis", *CARACTERISTICAS])

print("Origen:", origen)
datos.head()

Origen: data\wdbc.data


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 3. Identificación del dataset

In [3]:
conteo = datos["diagnosis"].value_counts()

print(f"Registros:               {datos.shape[0]}")
print(f"Columnas:                {datos.shape[1]}")
print(f"Características entrada: {len(CARACTERISTICAS)}")
print("Variable objetivo:       diagnosis (M = maligno, B = benigno)")
print(f"Valores nulos:           {datos.isna().sum().sum()}")
print(f"ID repetidos:            {datos['id'].duplicated().sum()}")
print(f"Tipos en las entradas:   {datos[CARACTERISTICAS].dtypes.unique().tolist()}")

distribucion = pd.DataFrame(
    {"Registros": conteo, "Porcentaje": (conteo / len(datos) * 100).round(1)}
).rename(index={"B": "Benigno (B)", "M": "Maligno (M)"})
display(distribucion)

Registros:               569
Columnas:                32
Características entrada: 30
Variable objetivo:       diagnosis (M = maligno, B = benigno)
Valores nulos:           0
ID repetidos:            0
Tipos en las entradas:   [dtype('float64')]


,Registros,Porcentaje
diagnosis,,
Benigno (B),357,62.7
Maligno (M),212,37.3


**Qué miden las variables.** Cada registro describe los núcleos celulares de una imagen de punción con aguja fina. Hay 10 medidas y cada una aparece de tres formas: promedio (`_mean`), error estándar (`_se`) y promedio de los tres valores más grandes (`_worst`). Por eso son 30 características.

| Medida | Significado |
|---|---|
| `radius` | Distancia media del centro al perímetro |
| `texture` | Desviación estándar de los valores de gris |
| `perimeter` | Perímetro del núcleo |
| `area` | Área del núcleo |
| `smoothness` | Variación local en la longitud de los radios |
| `compactness` | perímetro² / área − 1 |
| `concavity` | Severidad de las partes cóncavas del contorno |
| `concave_points` | Número de partes cóncavas del contorno |
| `symmetry` | Simetría del núcleo |
| `fractal_dimension` | Aproximación de la "línea de costa" − 1 |

La columna `id` solo identifica al paciente. No aporta información para clasificar y se elimina.

**Lo que esto implica.** Las clases no están balanceadas: hay más benignos que malignos. Eso vuelve engañosa a la exactitud por sí sola, como se comprueba en la sección 9.

In [4]:
rangos = datos[CARACTERISTICAS].describe().T[["min", "mean", "max", "std"]]
display(rangos.loc[["area_worst", "perimeter_mean", "radius_mean", "smoothness_mean", "fractal_dimension_se"]].round(4))

,min,mean,max,std
area_worst,185.2000,880.5831,4254.0000,569.3570
perimeter_mean,43.7900,91.9690,188.5000,24.2990
radius_mean,6.9810,14.1273,28.1100,3.5240
smoothness_mean,0.0526,0.0964,0.1634,0.0141
fractal_dimension_se,0.0009,0.0038,0.0298,0.0026


Las escalas son muy distintas: `area_worst` llega a miles y `fractal_dimension_se` no pasa de centésimas. KNN y SVM usan distancias, así que sin estandarizar `area` dominaría el resultado.

## 4. Separación de entradas y variable objetivo

In [5]:
X = datos[CARACTERISTICAS]
y = datos["diagnosis"].map({"B": 0, "M": 1})  # 1 = maligno, la clase positiva

print("X:", X.shape)
print("y:", y.shape, "| malignos:", int(y.sum()), "| benignos:", int((y == 0).sum()))

X: (569, 30)
y: (569,) | malignos: 212 | benignos: 357


## 5. División en entrenamiento y prueba

Se hace **una sola vez** y los tres modelos usan estos mismos conjuntos. Usamos 80 % / 20 % y `stratify=y` para conservar la proporción de malignos en ambos lados.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEMILLA
)

division = pd.DataFrame(
    {
        "Registros": [len(y_train), len(y_test)],
        "Malignos": [int(y_train.sum()), int(y_test.sum())],
        "Benignos": [int((y_train == 0).sum()), int((y_test == 0).sum())],
        "% malignos": [round(y_train.mean() * 100, 1), round(y_test.mean() * 100, 1)],
    },
    index=["Entrenamiento", "Prueba"],
)
division

,Registros,Malignos,Benignos,% malignos
Entrenamiento,455,170,285,37.4
Prueba,114,42,72,36.8


## 6. Preprocesamiento: estandarización

`StandardScaler` deja cada característica con media 0 y desviación estándar 1. **Solo aprende del conjunto de entrenamiento**; si viera los datos de prueba, se filtraría información y los resultados saldrían inflados.

In [7]:
escalador = StandardScaler().fit(X_train)
X_train_esc = pd.DataFrame(escalador.transform(X_train), columns=CARACTERISTICAS, index=X_train.index)
X_test_esc = pd.DataFrame(escalador.transform(X_test), columns=CARACTERISTICAS, index=X_test.index)

efecto = pd.DataFrame(
    {
        "Media antes": X_train[["area_worst", "smoothness_mean"]].mean(),
        "Desv. antes": X_train[["area_worst", "smoothness_mean"]].std(ddof=0),
        "Media después": X_train_esc[["area_worst", "smoothness_mean"]].mean(),
        "Desv. después": X_train_esc[["area_worst", "smoothness_mean"]].std(ddof=0),
    }
)
efecto.round(4) + 0.0  # sumar 0.0 evita que se muestre -0.0

,Media antes,Desv. antes,Media después,Desv. después
area_worst,890.5692,581.7059,0.0,1.0
smoothness_mean,0.0960,0.0143,0.0,1.0


Esta celda muestra el efecto. Para entrenar metemos el escalador en un `Pipeline`, de modo que dentro de la validación cruzada se ajusta de nuevo en cada partición y tampoco ve los datos con los que se valida.